In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/voltpath/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import col, trim, initcap, upper, to_date


# --- Silver: stations ---
# station_name: trimmed | city: trimmed + each word capitalised | site_type: trimmed + upper | opened_on: DATE
bronze_stations = spark.table(f"{catalog}.bronze.stations")

silver_stations = bronze_stations.select(
    col("station_id"),
    trim(col("station_name")).alias("station_name"),
    initcap(trim(col("city"))).alias("city"),
    upper(trim(col("site_type"))).alias("site_type"),
    to_date(col("opened_on"), "yyyy-MM-dd").alias("opened_on")
)

silver_stations.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.stations")

# --- Silver: tariffs ---
# time_band: trimmed + upper | rate_per_kwh & idle_fee_per_min: DECIMAL(10,2) | free_idle_minutes: INT
bronze_tariffs = spark.table(f"{catalog}.bronze.tariffs")

silver_tariffs = bronze_tariffs.select(
    upper(trim(col("connector_type"))).alias("connector_type"),
    upper(trim(col("time_band"))).alias("time_band"),
    col("rate_per_kwh").cast("decimal(10,2)").alias("rate_per_kwh"),
    col("idle_fee_per_min").cast("decimal(10,2)").alias("idle_fee_per_min"),
    col("free_idle_minutes").cast("int").alias("free_idle_minutes")
)

silver_tariffs.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.tariffs")


In [0]:
from pyspark.sql.functions import col, upper, trim, to_date, row_number
from pyspark.sql.window import Window

# --- Read and transform bronze.chargers ---
bronze_chargers = spark.table(f"{catalog}.bronze.chargers")

# Transform: extract connector fields, cast to required types
# last_changed_snapshot_date = snapshot that inserted or last changed the row
chargers_transformed = bronze_chargers.select(
    col("charger_id"),
    col("station_id"),
    upper(trim(col("connector.type"))).alias("connector_type"),
    col("connector.max_kw").cast("decimal(6,1)").alias("max_kw"),
    col("is_active").cast("boolean").alias("is_active"),
    to_date(col("commissioned_on"), "yyyy-MM-dd").alias("commissioned_on"),
    to_date(col("snapshot_date"), "yyyy-MM-dd").alias("last_changed_snapshot_date")
)

# One row per charger: keep only the latest snapshot's values
w = Window.partitionBy("charger_id").orderBy(col("last_changed_snapshot_date").desc())
latest_chargers = (chargers_transformed
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Create target table if not exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.chargers (
    charger_id STRING,
    station_id STRING,
    connector_type STRING,
    max_kw DECIMAL(6,1),
    is_active BOOLEAN,
    commissioned_on DATE,
    last_changed_snapshot_date DATE
) USING DELTA
""")

# Temp view for MERGE source
latest_chargers.createOrReplaceTempView("latest_chargers_tempview")

# MERGE INTO: update only if a value other than last_changed_snapshot_date changed
# IS DISTINCT FROM is null-safe (unlike <>)
spark.sql(f"""
MERGE INTO {catalog}.silver.chargers AS t
USING latest_chargers_tempview AS s
ON t.charger_id = s.charger_id
WHEN MATCHED AND (
    t.station_id IS DISTINCT FROM s.station_id OR
    t.connector_type IS DISTINCT FROM s.connector_type OR
    t.max_kw IS DISTINCT FROM s.max_kw OR
    t.is_active IS DISTINCT FROM s.is_active OR
    t.commissioned_on IS DISTINCT FROM s.commissioned_on
) THEN UPDATE SET
    t.station_id = s.station_id,
    t.connector_type = s.connector_type,
    t.max_kw = s.max_kw,
    t.is_active = s.is_active,
    t.commissioned_on = s.commissioned_on,
    t.last_changed_snapshot_date = s.last_changed_snapshot_date
WHEN NOT MATCHED THEN INSERT (
    charger_id, station_id, connector_type, max_kw, is_active, commissioned_on, last_changed_snapshot_date
) VALUES (
    s.charger_id, s.station_id, s.connector_type, s.max_kw, s.is_active, s.commissioned_on, s.last_changed_snapshot_date
)
""")

# Enable Change Data Feed before Batch 2 load
spark.sql(f"ALTER TABLE {catalog}.silver.chargers SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

In [0]:
from pyspark.sql.functions import (
    col, trim, lower, upper, initcap, to_date, regexp_replace,
    substring, length, greatest, lit, coalesce, when
)

catalog = "vp_rp20040126"

# --- Step 1: Read and standardize bronze.member ---
bronze_members = spark.table(f"{catalog}.bronze.member")

standardized = bronze_members.select(
    col("member_id"),
    trim(col("full_name")).alias("full_name"),
    lower(trim(col("email"))).alias("email"),
    # phone: digits only, last 10
    substring(
        regexp_replace(col("phone"), "[^0-9]", ""),
        greatest(length(regexp_replace(col("phone"), "[^0-9]", "")) - lit(9), lit(1)),
        10
    ).alias("phone"),
    initcap(trim(col("city"))).alias("city"),       # same as silver.stations
    upper(trim(col("plan"))).alias("plan"),
    when(col("joined_on").contains("/"), to_date(col("joined_on"), "dd/MM/yyyy"))
     .otherwise(to_date(col("joined_on"), "yyyy-MM-dd")).alias("joined_on"),
    # effective_date: 2026-04-01 for snapshots, change_effective_date for changes
    when(col("change_effective_date").isNull(), to_date(lit("2026-04-01")))
     .otherwise(to_date(col("change_effective_date"), "yyyy-MM-dd")).alias("effective_date")
)

# Deduplicate identical rows (req 1: identical rows count once)
standardized = standardized.dropDuplicates(
    ["member_id", "full_name", "email", "phone", "city", "plan", "joined_on", "effective_date"]
)

standardized.createOrReplaceTempView("std_members_scd2")

# --- Step 2: SCD Type 2 processing ---
# a) Order by member, effective_date; use LAG to detect no-op changes
# b) Filter out no-op changes (all standardized values match previous — req 5)
# c) Detect version boundaries: plan or city changed → new version (req 3)
# d) Assign version groups via cumulative sum
# e) Aggregate per version: effective_from = MIN(date), attributes = last value (req 4: overwrite)
# f) Add effective_to (next version start) and is_current (req: NULL effective_to = current)
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.silver.members AS
WITH ordered AS (
    SELECT
        member_id, full_name, email, phone, city, plan, joined_on, effective_date,
        ROW_NUMBER() OVER (PARTITION BY member_id ORDER BY effective_date) AS rn,
        LAG(full_name)  OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_full_name,
        LAG(email)      OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_email,
        LAG(phone)      OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_phone,
        LAG(city)       OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_city,
        LAG(plan)       OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_plan,
        LAG(joined_on)  OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_joined_on
    FROM std_members_scd2
),
changed AS (
    SELECT member_id, full_name, email, phone, city, plan, joined_on, effective_date
    FROM ordered
    WHERE rn = 1
       OR full_name  IS DISTINCT FROM prev_full_name
       OR email      IS DISTINCT FROM prev_email
       OR phone      IS DISTINCT FROM prev_phone
       OR city       IS DISTINCT FROM prev_city
       OR plan       IS DISTINCT FROM prev_plan
       OR joined_on  IS DISTINCT FROM prev_joined_on
),
versioned AS (
    SELECT
        member_id, full_name, email, phone, city, plan, joined_on, effective_date,
        ROW_NUMBER() OVER (PARTITION BY member_id ORDER BY effective_date) AS rn2,
        LAG(plan) OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_plan2,
        LAG(city) OVER (PARTITION BY member_id ORDER BY effective_date) AS prev_city2
    FROM changed
),
versioned2 AS (
    SELECT
        member_id, full_name, email, phone, city, plan, joined_on, effective_date,
        CASE
            WHEN rn2 = 1 THEN 1
            WHEN plan IS DISTINCT FROM prev_plan2 OR city IS DISTINCT FROM prev_city2 THEN 1
            ELSE 0
        END AS is_new_version
    FROM versioned
),
grouped AS (
    SELECT
        member_id, full_name, email, phone, city, plan, joined_on, effective_date,
        SUM(is_new_version) OVER (
            PARTITION BY member_id ORDER BY effective_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS version_group
    FROM versioned2
),
aggregated AS (
    SELECT
        member_id,
        max_by(full_name,  effective_date) AS full_name,
        max_by(email,      effective_date) AS email,
        max_by(phone,      effective_date) AS phone,
        max_by(city,       effective_date) AS city,
        max_by(plan,       effective_date) AS plan,
        max_by(joined_on,  effective_date) AS joined_on,
        MIN(effective_date) AS effective_from
    FROM grouped
    GROUP BY member_id, version_group
)
SELECT
    member_id,
    full_name,
    email,
    phone,
    city,
    plan,
    joined_on,
    effective_from,
    LEAD(effective_from) OVER (PARTITION BY member_id ORDER BY effective_from) AS effective_to,
    CASE
        WHEN LEAD(effective_from) OVER (PARTITION BY member_id ORDER BY effective_from) IS NULL THEN true
        ELSE false
    END AS is_current
FROM aggregated
ORDER BY member_id, effective_from
""")

display(spark.table(f"{catalog}.silver.members"))